# Grammar Scoring Engine for spoken audio

Predict a grammar score (1 to 5) for a 45 to 60 second spoken answer.
769 training recordings, 216 test recordings.

## Report

> **TO WRITE (in your own words, then delete this note).** Keep each part to a few sentences.
> The numbers you need are printed by the cells below.
>
> 1. **Approach.** What the pipeline does, in order (table below).
> 2. **Preprocessing.** 16 kHz mono, 30 second windows, zero-score files left out and why.
> 3. **What the data taught us.** Speakers repeat in training but not in test (section 4), and the short
>    test recordings answer questions that are rare or absent in training (section 3.5).
> 4. **Validation.** Speaker folds, short recordings weighted up, and why the leaderboard still had to
>    be used as a check for the last choices.
> 5. **Results.** Tables in section 6, final scores and the training RMSE in section 7, the
>    leaderboard history in section 8.
> 6. **What did not work** and **limits**: sections 8 and 9.
> 7. **Biggest single gain.** Fine-tuning DeBERTa on the transcripts (section 5.1), against frozen embeddings.

## Pipeline at a glance

| Step | What | Output |
|---|---|---|
| 1 | Whisper reads each recording | a transcript and a 1,280 number audio embedding |
| 2 | Two text models read the transcript, made topic neutral | two text embeddings |
| 3 | Simple and grammar specific measurements | about 34 small features |
| 4 | Two language models are asked to grade the grammar | their internal state at the moment they answer |
| 5 | DeBERTa-v3-base is fine-tuned on the transcripts to predict the score | one prediction per file |
| 6 | One model per feature block predicts the score | seven predictions per file |
| 7 | A small linear model blends them | one score per file |

## 1. Setup

In [ ]:
import glob, re, gc, time, zlib, difflib, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import librosa
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from scipy.stats import pearsonr
from scipy.sparse.csgraph import connected_components
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold

warnings.filterwarnings("ignore")

SEED = 42
SEEDS = (0, 1, 2)               # every cross validation is repeated over three shuffles
SR = 16000                      # Whisper works on 16 kHz mono audio
CHUNK = 30 * SR                 # Whisper sees 30 s at a time
ASR_MODEL = "openai/whisper-large-v3-turbo"
TEXT_MODELS = {"mpnet": "sentence-transformers/all-mpnet-base-v2", "bge": "BAAI/bge-large-en-v1.5"}
VOICE_MODEL = "microsoft/wavlm-large"   # used only to tell speakers apart, never to predict scores
INPUT_ROOT = "/kaggle/input"
WORK = Path("/kaggle/working")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)

In [ ]:
# Slow steps save their results to WORK. If an earlier run's files are attached as an input,
# copy them over so those steps are skipped. A zip that Kaggle left packed is opened as well.
import zipfile
CACHE_PATTERNS = ["audio_emb_*.npy", "transcripts_*.csv", "feats_*.csv", "voice_*.npy", "wavlm_*.npy",
                  "text_*.npy", "llm_*.npy", "judge_*.npz", "ft_*.npz",
                  "submission_c_ft_added.csv", "submission_c3_base_2runs.csv"]   # last two: only for the check in section 7
is_cache = lambda name: any(Path(name).match(p) for p in CACHE_PATTERNS)
restored = []
for src in glob.glob(f"{INPUT_ROOT}/**/*", recursive=True):
    if src.lower().endswith(".zip"):
        with zipfile.ZipFile(src) as zf:
            for member in zf.namelist():
                dst = WORK / Path(member).name
                if is_cache(dst.name) and not dst.exists():
                    dst.write_bytes(zf.read(member)); restored.append(dst.name)
    elif is_cache(Path(src).name) and not (WORK / Path(src).name).exists():
        shutil.copy(src, WORK / Path(src).name); restored.append(Path(src).name)
print("copied from attached inputs:", sorted(restored) if restored else "nothing")
present = sorted(p.name for p in WORK.iterdir() if is_cache(p.name))
print(f"cache files available ({len(present)}):", present if present else "none, everything will be computed (about an hour)")

## 2. Data

In [ ]:
def find_one(name):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    assert hits, f"{name} not found under {INPUT_ROOT}. Add the competition data to this notebook."
    return hits[0]

train = pd.read_csv(find_one("train.csv"))
test = pd.read_csv(find_one("test.csv"))
ID_COL, TARGET = train.columns[0], train.columns[1]
test = test[[ID_COL]].copy()              # the label column in test.csv is random filler
# Note: sample_submission.csv lists different file names than test.csv, so it is not used.

def stem(name):
    name = Path(str(name)).name
    return name[:-4] if name.lower().endswith(".wav") else name

wav_index = {}
for p in glob.glob(f"{INPUT_ROOT}/**/*.wav", recursive=True):
    wav_index.setdefault(stem(p), []).append(p)

def resolve(name, split):
    hits = wav_index.get(stem(name), [])
    if len(hits) > 1:                     # same file name in the train and test folders
        hits = [h for h in hits if split in h.lower()] or hits
    return hits[0] if hits else None

train["path"] = [resolve(n, "train") for n in train[ID_COL]]
test["path"] = [resolve(n, "test") for n in test[ID_COL]]
assert train["path"].notna().all() and test["path"].notna().all(), "some audio files were not found"
print(train.shape, test.shape, "| id:", ID_COL, "| target:", TARGET)

ax = train[TARGET].value_counts().sort_index().plot(kind="bar", figsize=(6, 3), color="steelblue")
ax.set_xlabel("grammar score"); ax.set_ylabel("files"); ax.set_title("Training label distribution")
plt.tight_layout(); plt.show()

## 3. Features

### 3.1 Whisper: transcript and audio embedding

One model does two jobs. Its **encoder** turns the audio into a sequence of vectors, one per 20 ms.
Their average over time is the audio embedding. Its **decoder** writes the transcript.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def load_audio(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
    except Exception as e:
        print("could not read", path, "->", e)
        y = np.zeros(SR, dtype=np.float32)
    return y if len(y) >= SR else np.pad(y, (0, SR - len(y)))

def run_whisper(df, split):
    emb_file, txt_file = WORK / f"audio_emb_{split}.npy", WORK / f"transcripts_{split}.csv"
    if emb_file.exists() and txt_file.exists():
        cached = pd.read_csv(txt_file, keep_default_na=False)
        if cached[ID_COL].astype(str).tolist() == df[ID_COL].astype(str).tolist():
            print(split, "whisper outputs loaded from cache")
            return np.load(emb_file), cached["text"].tolist(), cached["duration"].values

    processor = WhisperProcessor.from_pretrained(ASR_MODEL)
    asr = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL).to(DEVICE).eval()
    if DEVICE == "cuda":
        asr = asr.half()                               # half precision: faster, less GPU memory
    gen_kwargs = {} if ASR_MODEL.endswith(".en") else {"language": "en", "task": "transcribe"}

    embs, texts, durs = [], [], []
    for path in tqdm(df["path"], desc=f"whisper {split}"):
        y = load_audio(path)
        chunks = [y[i:i + CHUNK] for i in range(0, len(y), CHUNK)]
        if len(chunks) > 1 and len(chunks[-1]) < SR:   # drop a tail shorter than 1 s
            chunks = chunks[:-1]
        feats = processor.feature_extractor(chunks, sampling_rate=SR, return_tensors="pt").input_features
        feats = feats.to(DEVICE, dtype=asr.dtype)
        with torch.no_grad():
            hidden = asr.get_encoder()(feats).last_hidden_state.float()
            ids = asr.generate(feats, **gen_kwargs)
        # Each chunk is padded to 30 s = 1500 encoder frames (320 samples each). Average only the
        # frames that cover real audio, and weight chunks by their length.
        total, emb = 0, 0
        for h, c in zip(hidden, chunks):
            n_frames = min(h.shape[0], int(np.ceil(len(c) / 320)))
            emb = emb + h[:n_frames].mean(0) * len(c)
            total += len(c)
        embs.append((emb / total).cpu().numpy())
        texts.append(" ".join(t.strip() for t in processor.batch_decode(ids, skip_special_tokens=True)))
        durs.append(len(y) / SR)

    del asr
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    embs, durs = np.stack(embs), np.array(durs)
    np.save(emb_file, embs)
    pd.DataFrame({ID_COL: df[ID_COL], "text": texts, "duration": durs}).to_csv(txt_file, index=False)
    return embs, texts, durs

Xa_tr, train["text"], train["duration"] = run_whisper(train, "train")
Xa_te, test["text"], test["duration"] = run_whisper(test, "test")
print("audio embeddings:", Xa_tr.shape, Xa_te.shape)
print("duration (s):", train["duration"].describe()[["min", "mean", "max"]].round(1).to_dict())
for label, part in [("LOW", train[train[TARGET] > 0].nsmallest(2, TARGET)), ("HIGH", train.nlargest(2, TARGET))]:
    for _, r in part.iterrows():
        print(f"\n[{label} {r[TARGET]}] {r['text'][:300]}")

### 3.2 Text embeddings

Two pretrained text models read the transcript. For each, the token vectors are averaged
(ignoring padding) into one vector per recording.

In [ ]:
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def embed_texts(key, texts, split, bs=16):
    f = WORK / f"text_{key}_{split}.npy"
    if f.exists():
        return np.load(f)
    tok = AutoTokenizer.from_pretrained(TEXT_MODELS[key])
    model = AutoModel.from_pretrained(TEXT_MODELS[key]).to(DEVICE).eval()
    texts, out = [str(t) for t in texts], []
    for i in tqdm(range(0, len(texts), bs), desc=f"{key} {split}"):
        b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        h = model(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())            # mean over real tokens
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    out = np.vstack(out)
    np.save(f, out)
    return out

T_tr = {k: embed_texts(k, train["text"], "train") for k in TEXT_MODELS}
T_te = {k: embed_texts(k, test["text"], "test") for k in TEXT_MODELS}
print({k: v.shape for k, v in T_tr.items()})

### 3.3 Small features

About 34 numbers per recording, in four groups:

* **Counts**: words, speaking rate, sentence length, duration.
* **Vocabulary and repetition**: vocabulary size corrected for length, repeated word pairs.
* **Grammar**: share of sentences a grammar classifier accepts, how much a correction model
  rewrites, how surprising the text is to a language model.
* **Pauses**: silence statistics measured on the audio.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM

def sentences(text, max_n=40):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if len(s.split()) >= 3][:max_n]

def count_features(text, dur):
    text = str(text)
    w = re.findall(r"[a-z']+", text.lower())
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    n = len(w)
    return {"n_words": n, "words_per_sec": n / max(dur, 1e-6), "unique_word_ratio": len(set(w)) / max(n, 1),
            "words_per_sentence": n / max(len(sents), 1),
            "mean_word_length": float(np.mean([len(x) for x in w])) if n else 0.0, "duration": dur}

SUBORD = {"because", "although", "though", "since", "unless", "while", "whereas", "which", "who", "whom",
          "whose", "that", "if", "when", "where", "after", "before", "until"}
def lex_features(text):
    text = str(text)
    w = re.findall(r"[a-z']+", text.lower())
    n = max(len(w), 1)
    counts = pd.Series(w).value_counts() if w else pd.Series(dtype=int)
    bi, tri = list(zip(w, w[1:])), list(zip(w, w[1:], w[2:]))
    sl = np.array([len(s.split()) for s in re.split(r"[.!?]+", text) if s.strip()] or [0])
    raw = text.encode()
    return {
        "lex_guiraud": len(counts) / np.sqrt(n),                       # vocabulary size, corrected for length
        "lex_hapax": float((counts == 1).sum()) / n,                   # words used exactly once
        "lex_long_words": sum(len(x) >= 7 for x in w) / n,
        "rep_bigram": 1 - len(set(bi)) / max(len(bi), 1),              # repeated word pairs
        "rep_trigram": 1 - len(set(tri)) / max(len(tri), 1),
        "rep_immediate": sum(a == b for a, b in bi) / n,               # "the the", "I I"
        "rep_compress": len(zlib.compress(raw)) / max(len(raw), 1),    # low value = very repetitive text
        "syn_subord_per_sent": sum(x in SUBORD for x in w) / len(sl),  # rough sign of complex sentences
        "syn_commas": text.count(",") / n,
        "sent_len_std": float(sl.std()), "sent_len_max": float(sl.max()), "sent_short": float((sl <= 3).mean()),
    }

@torch.no_grad()
def cola_features(texts, name="textattack/roberta-base-CoLA", bs=64):
    # A classifier trained to say whether a sentence is grammatically acceptable.
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
    rows = []
    for t in tqdm(texts, desc="acceptability"):
        ss = sentences(t)
        if not ss:
            rows.append({"cola_mean": np.nan, "cola_min": np.nan, "cola_bad_share": np.nan}); continue
        p = []
        for i in range(0, len(ss), bs):
            b = tok(ss[i:i + bs], padding=True, truncation=True, max_length=96, return_tensors="pt").to(DEVICE)
            p.append(torch.softmax(mdl(**b).logits.float(), -1)[:, 1].cpu().numpy())      # P(acceptable)
        p = np.concatenate(p)
        rows.append({"cola_mean": p.mean(), "cola_min": p.min(), "cola_bad_share": (p < 0.5).mean()})
    del mdl
    return pd.DataFrame(rows)

@torch.no_grad()
def gec_features(texts, name="vennify/t5-base-grammar-correction", bs=32):
    # A model that rewrites a sentence with corrected grammar. We measure how much it had to change.
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
    norm = lambda s: re.findall(r"[a-z']+", s.lower())
    owner, sents = [], []
    for i, t in enumerate(texts):
        for s in sentences(t, max_n=25):
            owner.append(i); sents.append(s)
    order = np.argsort([len(s) for s in sents])                        # similar lengths together = faster
    fixed = [None] * len(sents)
    for i in tqdm(range(0, len(sents), bs), desc="correction"):
        idx = order[i:i + bs]
        b = tok(["grammar: " + sents[j] for j in idx], padding=True, truncation=True, max_length=96, return_tensors="pt").to(DEVICE)
        out = mdl.generate(input_ids=b["input_ids"], attention_mask=b["attention_mask"], max_new_tokens=96, num_beams=1, do_sample=False)
        for j, o in zip(idx, tok.batch_decode(out, skip_special_tokens=True)):
            fixed[j] = o
    edit = np.array([1 - difflib.SequenceMatcher(None, norm(a), norm(b)).ratio() for a, b in zip(sents, fixed)])
    g = pd.DataFrame({"owner": owner, "edit": edit}).groupby("owner")["edit"]
    res = pd.DataFrame({"gec_edit_mean": g.mean(), "gec_edit_max": g.max(), "gec_changed_share": g.apply(lambda v: (v > 0).mean())})
    del mdl
    return res.reindex(range(len(texts))).reset_index(drop=True)

@torch.no_grad()
def lm_features(texts, name="gpt2"):
    # How surprising the transcript is to a language model. Unnatural word order raises it.
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name).to(DEVICE).eval()
    out = []
    for t in tqdm(texts, desc="language model"):
        ids = tok(str(t), return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
        out.append(float(mdl(ids, labels=ids).loss) if ids.shape[1] >= 2 else np.nan)
    del mdl
    return pd.DataFrame({"lm_surprise": out})

PAUSE_KEYS = ["pau_speech_ratio", "pau_per_min", "pau_mean", "pau_max", "pau_time_ratio", "pau_run_mean", "pau_lead", "pau_energy_sd"]
def pause_features(path):
    y_ = load_audio(path)
    dur = len(y_) / SR
    iv = librosa.effects.split(y_, top_db=30, frame_length=1024, hop_length=256)      # stretches of sound
    if len(iv) == 0:
        return dict.fromkeys(PAUSE_KEYS, np.nan)
    gaps = (iv[1:, 0] - iv[:-1, 1]) / SR
    gaps = gaps[gaps >= 0.3]                                                            # silences of 0.3 s or more
    speech = (iv[:, 1] - iv[:, 0]).sum() / SR
    rms = librosa.feature.rms(y=y_, frame_length=1024, hop_length=256)[0]
    return dict(zip(PAUSE_KEYS, [speech / dur, len(gaps) / dur * 60, gaps.mean() if len(gaps) else 0.0,
                                 gaps.max() if len(gaps) else 0.0, gaps.sum() / dur, speech / (len(gaps) + 1),
                                 iv[0, 0] / SR, float(np.std(20 * np.log10(rms + 1e-6)))]))

def build_small(df, split):
    f = WORK / f"feats_{split}.csv"
    if f.exists():
        slow = pd.read_csv(f)
        print(split, "model based features loaded from cache")
    else:
        texts = df["text"].astype(str).tolist()
        parts = [pd.DataFrame([lex_features(t) for t in texts])]
        for fn in (cola_features, gec_features, lm_features):
            parts.append(fn(texts))
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        parts.append(pd.DataFrame(Parallel(n_jobs=4, prefer="threads")(delayed(pause_features)(p) for p in df["path"])))
        slow = pd.concat(parts, axis=1)
        slow.to_csv(f, index=False)
    counts = pd.DataFrame([count_features(t, d) for t, d in zip(df["text"], df["duration"])])
    out = pd.concat([counts, slow], axis=1)
    out["rate_articulation"] = out["n_words"] / (out["duration"] * out["pau_speech_ratio"]).clip(lower=1)   # words per second of actual speech
    return out

F_tr, F_te = build_small(train, "train"), build_small(test, "test")
print("small features:", F_tr.shape)

### 3.4 Voice fingerprints (for validation only)

The lower layers of WavLM describe who is speaking. Averaged over time they give a fingerprint of the
voice. It is **not** used to predict scores. It is used to find recordings by the same speaker, so that
cross validation can keep a speaker's recordings together.

In [ ]:
from transformers import AutoFeatureExtractor

def run_voice(df, split):
    f, full = WORK / f"voice_{split}.npy", WORK / f"wavlm_{split}.npy"
    if f.exists():
        return np.load(f)
    if full.exists():                                   # all-layer file from an earlier run
        v = np.load(full).astype(np.float32)[:, 2:9].mean(1)
    else:
        fe = AutoFeatureExtractor.from_pretrained(VOICE_MODEL)
        wav = AutoModel.from_pretrained(VOICE_MODEL).to(DEVICE).eval()
        v = []
        for p in tqdm(df["path"], desc=f"voice {split}"):
            x = fe(load_audio(p), sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
            with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
                hs = wav(x, output_hidden_states=True).hidden_states
            v.append(torch.stack([h[0].float().mean(0) for h in hs[2:9]]).mean(0).cpu().numpy())
        v = np.stack(v)
        del wav
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    np.save(f, v)
    return v

voice_tr, voice_te = run_voice(train, "train"), run_voice(test, "test")
print("voice fingerprints:", voice_tr.shape, voice_te.shape)

### 3.5 Topics: the short test recordings answer new questions

Transcripts are grouped into 10 topics. The short test recordings are mostly about topics that are rare
or absent among short training recordings. A model can learn in training that some topics score lower,
and that does not carry over to new questions. So the two text embeddings are made **topic neutral**:
each file's vector minus the average vector of its topic. This uses no labels, only the inputs.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

K = 10
E = np.vstack([T_tr["bge"], T_te["bge"]]).astype(np.float32)
E /= np.linalg.norm(E, axis=1, keepdims=True)
topic_all = KMeans(K, n_init=10, random_state=0).fit_predict(E)
n_tr = len(train)
texts_all = pd.concat([train["text"], test["text"]]).astype(str).values
tfidf = TfidfVectorizer(stop_words="english", min_df=3, max_features=5000).fit(texts_all)
M, vocab = tfidf.transform(texts_all), np.array(tfidf.get_feature_names_out())
short_full = np.r_[train["duration"].values < 50, test["duration"].values < 50]
is_test_row = np.r_[np.zeros(n_tr, bool), np.ones(len(test), bool)]
y_full = np.r_[train[TARGET].values.astype(float), np.full(len(test), np.nan)]
rows = []
for k in np.unique(topic_all):
    m = topic_all == k
    lab = y_full[m & ~is_test_row]
    lab = lab[lab > 0]
    rows.append({"topic": k, "typical words": " ".join(vocab[np.asarray(M[m].mean(0)).ravel().argsort()[::-1][:5]]),
                 "train short": int((m & ~is_test_row & short_full).sum()), "test short": int((m & is_test_row & short_full).sum()),
                 "train long": int((m & ~is_test_row & ~short_full).sum()), "test long": int((m & is_test_row & ~short_full).sum()),
                 "train mean score": round(float(lab.mean()), 2) if len(lab) else np.nan})
topic_table = pd.DataFrame(rows).set_index("topic")
print(topic_table.to_string())

def center_by_topic(Xtr, Xte):
    # Subtract each topic's average vector (train and test files together, no labels used).
    X = np.vstack([Xtr, Xte]).astype(np.float32)
    out = X.copy()
    for k in np.unique(topic_all):
        m = topic_all == k
        out[m] -= X[m].mean(0)
    return out[:len(Xtr)], out[len(Xtr):]

Tc_tr, Tc_te = {}, {}
for key in TEXT_MODELS:
    Tc_tr[key], Tc_te[key] = center_by_topic(T_tr[key], T_te[key])

### 3.6 Two language models as grammar judges

Each model is given the rubric and a transcript and asked for a grade from 1 to 5. Its grade on its own
is only moderately accurate, so it is not used. What is used is the model's **internal state** at the
moment it is about to answer: a vector that summarises how it read the transcript in light of the rubric.
Two different models (Qwen2.5 7B and Phi-3.5 mini) give two such vectors.

In [ ]:
from transformers import AutoModelForCausalLM

RUBRIC = ("1: Struggles with proper sentence structure and syntax, limited control over simple grammatical structures and memorized sentence patterns.\n"
          "2: Limited understanding of sentence structure and syntax. Uses simple structures but consistently makes basic sentence structure and grammatical mistakes. May leave sentences incomplete.\n"
          "3: Decent grasp of sentence structure but makes errors in grammatical structure, or decent grasp of grammatical structure but makes errors in sentence syntax and structure.\n"
          "4: Strong understanding of sentence structure and syntax. Consistently good control of grammar. Occasional errors are minor and do not lead to misunderstandings.\n"
          "5: High grammatical accuracy and adept control of complex grammar. Seldom makes noticeable mistakes and handles complex language structures well.")

def judge_prompt(text):
    return ("You are an experienced examiner of spoken English. Below is an automatic transcript of a candidate's spoken "
            "answer. Punctuation and capitalisation were added by the transcription software, so ignore them. Rate only the "
            "candidate's grammar using this rubric.\n\n" + RUBRIC + "\n\nTranscript:\n" + str(text)[:2500] +
            "\n\nReply with a single digit from 1 to 5.")

JUDGES = {                                    # key: (model, needs both GPUs)
    "qwen25_rubric": ("Qwen/Qwen2.5-7B-Instruct", True),
    "phi35_rubric":  ("microsoft/Phi-3.5-mini-instruct", False),
}

@torch.no_grad()
def run_judge(key, bs=4):
    name, big = JUDGES[key]
    files = {s: WORK / f"judge_{key}_{s}.npz" for s in ("train", "test")}
    if key == "qwen25_rubric" and not files["train"].exists() and (WORK / "llm_hidden_train.npy").exists():
        for s in files:                       # same model and prompt, saved under an older name
            np.savez(files[s], probs=np.load(WORK / f"llm_probs_{s}.npy"), states=np.load(WORK / f"llm_hidden_{s}.npy"))
    if all(f.exists() for f in files.values()):
        return {s: np.load(f) for s, f in files.items()}
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"                 # the last position is then the answer slot in every row
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    two = big and DEVICE == "cuda"
    if two:
        assert torch.cuda.device_count() >= 2, "this model needs both T4 GPUs (accelerator: GPU T4 x2)"
    kw = {"device_map": "auto", "max_memory": {0: "9GiB", 1: "12GiB"}} if two else {}
    llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16 if DEVICE == "cuda" else torch.float32, **kw)
    llm = (llm if two else llm.to(DEVICE)).eval()
    dev = next(llm.parameters()).device
    digits = [tok.encode(str(d), add_special_tokens=False)[-1] for d in range(1, 6)]
    def prompt(text):
        user = judge_prompt(text)
        if getattr(tok, "chat_template", None):
            return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
        return user + "\nScore: "
    out = {}
    for s, df in (("train", train), ("test", test)):
        texts, probs, hid = df["text"].astype(str).tolist(), [], []
        for i in tqdm(range(0, len(texts), bs), desc=f"{key} {s}"):
            b = tok([prompt(t) for t in texts[i:i + bs]], return_tensors="pt", padding=True, truncation=True, max_length=1000).to(dev)
            o = llm(**b, output_hidden_states=True)
            probs.append(torch.softmax(o.logits[:, -1, digits].float(), -1).cpu().numpy())        # P(1)..P(5)
            n = len(o.hidden_states) - 1
            hid.append(np.stack([o.hidden_states[l][:, -1].float().cpu().numpy() for l in (n // 2, 3 * n // 4, n)], 1).astype(np.float16))
        np.savez(files[s], probs=np.vstack(probs), states=np.vstack(hid))
        out[s] = np.load(files[s])
    del llm, o
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out

judge_raw = {key: run_judge(key) for key in JUDGES}

## 4. Validation design

Three facts about the data shape how the model is validated.

1. **37 training files have score 0.** The rubric starts at 1, and the model finds no such files in
   the test set. They are left out of training.
2. **Speakers repeat in training but not in test.** With random folds, a model can recognise a voice
   and recall that speaker's other score. Folds are therefore built so that one speaker is never on
   both sides.
3. **Short recordings are 24% of training but 69% of test.** Scores are computed with short
   recordings weighted up to their share in the test set.
4. **Most short test recordings answer new questions** (section 3.5). Cross validation cannot fully
   imitate this, so for the last few choices the public leaderboard was used as a check (section 8).

The leaderboard score behaves like a mix of RMSE and (1 - Pearson), so both are reported, with their
average as a single number (`mix`, lower is better).

In [ ]:
y_all = train[TARGET].values.astype(float)
keep = y_all > 0                                        # fact 1: leave out the zero-score files
yk = y_all[keep]
print("training files used:", keep.sum(), "| left out (score 0):", (~keep).sum())

# Fact 3: weights that make the training mix of short and long recordings match the test mix.
short_tr = train["duration"].values[keep] < 50
short_te = test["duration"].values < 50
p_short = short_te.mean()
weights = np.where(short_tr, p_short / short_tr.mean(), (1 - p_short) / (1 - short_tr.mean()))
print(f"short recordings: {short_tr.mean():.2f} of train, {p_short:.2f} of test")

# Fact 2: compare voices. Cosine similarity of centred fingerprints.
mu_v = voice_tr[keep].mean(0)
unit = lambda X: (X - mu_v) / np.linalg.norm(X - mu_v, axis=1, keepdims=True)
v_tr, v_te = unit(voice_tr[keep]), unit(voice_te)
s_tt = v_tr @ v_tr.T
np.fill_diagonal(s_tt, -1)
nn_tr, nn_te = s_tt.max(1), (v_te @ v_tr.T).max(1)
nn_idx = s_tt.argmax(1)
same = nn_tr >= 0.85
print(f"training files with a near identical voice elsewhere in training: {same.mean():.2f}")
print(f"  score gap inside such pairs: {np.abs(yk - yk[nn_idx])[same].mean():.2f}"
      f" | between two random files: {np.abs(yk[:, None] - yk[None, :]).mean():.2f}")
print(f"share with a training voice at similarity 0.80 or more: train {np.mean(nn_tr >= 0.8):.2f} | test {np.mean(nn_te >= 0.8):.2f}")

fig, ax = plt.subplots(figsize=(6.5, 3))
bins = np.linspace(0, 1, 41)
ax.hist(nn_tr, bins=bins, alpha=0.6, density=True, label="training files")
ax.hist(nn_te, bins=bins, alpha=0.6, density=True, label="test files")
ax.set_xlabel("similarity to the closest training voice"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Speakers repeat inside training, test speakers are new")
plt.tight_layout(); plt.show()

# Speaker groups: link every pair of training files above the threshold, chains included.
np.fill_diagonal(s_tt, 1)
n_groups, GROUPS = connected_components(s_tt >= 0.80, directed=False)
print(f"speaker groups: {n_groups} | largest: {np.bincount(GROUPS).max()} files")

# Folds are also balanced by score band and by recording length.
bands = pd.Series(pd.qcut(yk, 4, labels=False, duplicates="drop")).astype(str) + pd.Series(short_tr).astype(str)
STRATA = pd.factorize(bands)[0]

# A second way to fold: a whole topic is held out at a time (section 3.5). It imitates answers to new questions.
topic_groups = topic_all[:len(train)][keep]
print(f"topics among the training files used: {len(np.unique(topic_groups))}")

## 5. Models and stacking

* **Ridge regression** on each large embedding. Linear, with a strong penalty chosen inside the
  training fold. With about 730 files and a thousand features, anything more flexible overfits.
* **Small boosted trees** on the small features, where a few features can interact.
* **Stacking**: each block predicts the score. A linear model with non negative weights learns how
  much to trust each block. It is trained only on predictions made for files the block models had not
  seen, and it also corrects ridge's habit of pulling predictions toward the average.

In [ ]:
ALPHAS = np.logspace(0, 5, 21)
def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))
def gbm():
    return HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)

def cv_predict(make, X, groups="speaker"):
    # Returns one row of out of fold predictions per shuffle.
    oof = np.zeros((len(SEEDS), len(yk)))
    for s, seed in enumerate(SEEDS):
        if groups == "speaker":
            splits = StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(X, STRATA, GROUPS)
        else:
            splits = StratifiedKFold(5, shuffle=True, random_state=seed).split(X, STRATA)
        for tr, va in splits:
            oof[s, va] = make().fit(X[tr], yk[tr]).predict(X[va])
    return oof

def score(oof):
    # Weighted to the test mix. Each shuffle is scored separately, then averaged.
    out = []
    for p in np.clip(np.atleast_2d(oof), 1, 5):
        rm = np.sqrt(np.average((p - yk) ** 2, weights=weights))
        c = np.cov(p, yk, aweights=weights)
        pe = c[0, 1] / np.sqrt(c[0, 0] * c[1, 1])
        out.append({"rmse": rm, "pearson": pe, "mix": 0.5 * rm + 0.5 * (1 - pe)})
    return pd.DataFrame(out).mean()

def fit_stack(parts):
    # parts = {name: (model maker, training matrix for the files used, test matrix)}
    base_oof = {n: np.clip(cv_predict(mk, Xtr), 1, 5) for n, (mk, Xtr, _) in parts.items()}
    stack_oof = np.zeros((len(SEEDS), len(yk)))
    for s in range(len(SEEDS)):
        Z = np.column_stack([o[s] for o in base_oof.values()])
        for tr, va in GroupKFold(5).split(Z, groups=GROUPS):
            stack_oof[s, va] = LinearRegression(positive=True).fit(Z[tr], yk[tr]).predict(Z[va])
    meta = LinearRegression(positive=True).fit(np.column_stack([o.mean(0) for o in base_oof.values()]), yk)
    fitted = {n: mk().fit(Xtr, yk) for n, (mk, Xtr, _) in parts.items()}
    def predict(mats):                                  # mats = {block name: feature matrix}
        cols = [np.clip(fitted[n].predict(mats[n]), 1, 5) for n in parts]
        return np.clip(meta.predict(np.column_stack(cols)), 1, 5)
    return {"oof": np.clip(stack_oof, 1, 5), "weights": dict(zip(parts, meta.coef_)), "intercept": float(meta.intercept_),
            "predict": predict, "test_pred": predict({n: parts[n][2] for n in parts})}

# Feature blocks. Gaps in the small features are filled with the training median.
med = F_tr[keep].median()
S_tr, S_te = F_tr.fillna(med).values, F_te.fillna(med).values
# Each judge contributes the stored layer that predicts best on its own (chosen under speaker folds).
judge_tr, judge_te = {}, {}
for key, z in judge_raw.items():
    H, Ht = z["train"]["states"].astype(np.float32), z["test"]["states"].astype(np.float32)
    alone = [score(cv_predict(ridge, H[keep][:, j]))["mix"] for j in range(H.shape[1])]
    j = int(np.argmin(alone))
    grade = z["train"]["probs"] @ np.arange(1, 6)
    print(f"{key}: own grade vs truth, Pearson {pearsonr(grade[keep], yk)[0]:.3f} | state alone, mix by layer "
          f"(middle, three quarters, last) {np.round(alone, 4)} -> using {['middle', 'three quarters', 'last'][j]}")
    judge_tr[key], judge_te[key] = H[:, j], Ht[:, j]

all_train = {"whisper audio": Xa_tr, "text mpnet (topic neutral)": Tc_tr["mpnet"], "text bge (topic neutral)": Tc_tr["bge"],
             "small features": S_tr, "judge qwen2.5": judge_tr["qwen25_rubric"], "judge phi3.5": judge_tr["phi35_rubric"]}
all_test = {"whisper audio": Xa_te, "text mpnet (topic neutral)": Tc_te["mpnet"], "text bge (topic neutral)": Tc_te["bge"],
            "small features": S_te, "judge qwen2.5": judge_te["qwen25_rubric"], "judge phi3.5": judge_te["phi35_rubric"]}
makers = {n: (gbm if n == "small features" else ridge) for n in all_train}
blocks = {n: (makers[n], all_train[n][keep], all_test[n]) for n in makers}

### 5.1 Fine-tuned DeBERTa

Every text model above is frozen: it turns a transcript into a vector and ridge is fitted on top. Here
the whole of DeBERTa-v3-base (184M parameters) is trained on our labels, with a small new output layer
that turns the transcript into one number. It can then learn which parts of a transcript matter for
grammar. This was the largest single gain (leaderboard 0.3440 to 0.3358).

* The model learns the standardised score (mean 0, spread 1 in its training fold) with squared error.
  5 epochs, learning rate 2e-5 with a short warm-up and a cosine decay, batches of 8, half precision.
* It is trained twice with 5 folds: once with speaker folds and once with topic folds. Every training
  file therefore has a prediction from a model that never saw that speaker, and one from a model that
  never saw that topic. These out of fold predictions are what the stack sees, so the stack never sees a
  prediction for a file the model was trained on.
* Test files get the average of all 10 fold models.
* The block model in the stack only looks these predictions up (`Lookup`): speaker fold predictions when
  the stack is scored with speaker folds, topic fold predictions when it is scored with topic folds.
* Results are cached in `ft_deberta_*.npz`. Without the cache, training takes about 12 minutes on a T4.

In [ ]:
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup

FT_MODEL = "microsoft/deberta-v3-base"
FT = dict(epochs=5, lr=2e-5, bs=8, max_len=512, warmup=0.1, wd=0.01)
FT_RUNS = 1        # 1 reproduces the 0.3358 submission. 2 averages a second run with other folds and seeds
                   # (better in cross validation, 0.3362 on the leaderboard).
texts_k = train["text"].astype(str).values[keep].tolist()
texts_te = test["text"].astype(str).tolist()
use_amp = DEVICE == "cuda"
ft_tok = None      # loaded only if a model has to be trained

@torch.no_grad()
def ft_predict(model, texts, bs=16):
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        x = ft_tok(texts[i:i + bs], truncation=True, max_length=FT["max_len"], padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out.append(model(**x).logits.float().squeeze(-1).cpu().numpy())
    return np.concatenate(out)

def ft_fold(tr, va, seed):
    # Train on the files in tr, predict the files in va and the test files.
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    # .float(): the checkpoint is stored in half precision, training needs full precision weights.
    model = AutoModelForSequenceClassification.from_pretrained(FT_MODEL, num_labels=1).float().to(DEVICE)
    mu, sd = yk[tr].mean(), yk[tr].std()
    opt = torch.optim.AdamW(model.parameters(), lr=FT["lr"], weight_decay=FT["wd"])
    steps = FT["epochs"] * int(np.ceil(len(tr) / FT["bs"]))
    sch = get_cosine_schedule_with_warmup(opt, int(FT["warmup"] * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for ep in range(FT["epochs"]):
        model.train()
        order = rng.permutation(tr)
        for i in range(0, len(order), FT["bs"]):
            b = order[i:i + FT["bs"]]
            x = ft_tok([texts_k[j] for j in b], truncation=True, max_length=FT["max_len"], padding=True, return_tensors="pt").to(DEVICE)
            t = torch.tensor((yk[b] - mu) / sd, dtype=torch.float32, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                pred = model(**x).logits.squeeze(-1)
            loss = F.mse_loss(pred.float(), t)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
    p_va = ft_predict(model, [texts_k[j] for j in va]) * sd + mu
    p_te = ft_predict(model, texts_te) * sd + mu
    del model, opt
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()
    return p_va, p_te

def ft_run(groups, tag, run):
    # One 5-fold run. Run 0 uses fold split 0 and seeds 0-4, run 1 uses split 1 and seeds 10-14.
    global ft_tok
    f = WORK / (f"ft_deberta_{tag}.npz" if run == 0 else f"ft_deberta_s{run}_{tag}.npz")
    if f.exists():
        z = np.load(f)
        return z["oof"], z["test"]
    if ft_tok is None:
        ft_tok = AutoTokenizer.from_pretrained(FT_MODEL)
    oof, te, t0 = np.zeros(len(yk)), [], time.time()
    for k, (tr, va) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=run).split(np.zeros(len(yk)), STRATA, groups)):
        p_va, p_te = ft_fold(tr, va, seed=10 * run + k)
        oof[va] = p_va
        te.append(p_te)
        print(f"  run {run}, {tag} fold {k}: rmse {np.sqrt(np.mean((np.clip(p_va, 1, 5) - yk[va]) ** 2)):.3f} | {time.time() - t0:.0f} s")
    np.savez(f, oof=oof, test=np.mean(te, 0))
    return oof, np.mean(te, 0)

speaker_groups = GROUPS
oof_s, oof_t, te_all = [], [], []
for run in range(FT_RUNS):
    a, ta = ft_run(speaker_groups, "speaker", run)
    b, tb = ft_run(topic_groups, "topic", run)
    oof_s.append(a); oof_t.append(b); te_all += [ta, tb]
ft_oof_spk, ft_oof_top, ft_test = np.mean(oof_s, 0), np.mean(oof_t, 0), np.mean(te_all, 0)
print(f"DeBERTa alone: speaker folds mix {score(ft_oof_spk)['mix']:.4f} | topic folds mix {score(ft_oof_top)['mix']:.4f}")

class Lookup:
    # The predictions are already out of fold, so the block model has nothing to fit. It returns
    # column 0 (speaker fold predictions) or, while the stack is scored with topic folds, column 1.
    def fit(self, X, y):
        return self
    def predict(self, X):
        return X[:, 1] if GROUPS is topic_groups else X[:, 0]

DEB = "deberta (fine-tuned)"
all_train[DEB] = np.full((len(train), 2), np.nan)       # no prediction for the zero-score files (not used)
all_train[DEB][keep] = np.column_stack([ft_oof_spk, ft_oof_top])
all_test[DEB] = np.column_stack([ft_test, ft_test])
makers[DEB] = Lookup
blocks[DEB] = (Lookup, all_train[DEB][keep], all_test[DEB])

def under_topic_folds(fn, *args):
    # Run fn with topic folds in place of speaker folds, and always switch back.
    global GROUPS
    try:
        GROUPS = topic_groups
        return fn(*args)
    finally:
        GROUPS = speaker_groups

## 6. Results

In [ ]:
# 6.1 Each block alone, under random folds and under speaker folds.
t0 = time.time()
rows = {}
for name, (mk, Xtr, _) in blocks.items():
    s = score(cv_predict(mk, Xtr, groups="speaker"))
    r = {"mix": np.nan} if mk is Lookup else score(cv_predict(mk, Xtr, groups="random"))   # DeBERTa: not trained with random folds
    t = score(under_topic_folds(cv_predict, mk, Xtr))
    rows[name] = {"mix, random folds": r["mix"], "mix, speaker folds": s["mix"], "mix, topic folds": t["mix"],
                  "rmse": s["rmse"], "pearson": s["pearson"]}
block_table = pd.DataFrame(rows).T
print("Each block alone (rmse and pearson are under speaker folds)")
print(block_table.round(4).to_string())
print(f"{time.time() - t0:.0f} s")

In [ ]:
# 6.2 The stack, and what each block contributes (leave one block out).
t0 = time.time()
final = fit_stack(blocks)
final_score = score(final["oof"])
abl = {"all blocks": final_score}
for name in blocks:
    abl[f"without {name}"] = score(fit_stack({k: v for k, v in blocks.items() if k != name})["oof"])
abl_table = pd.DataFrame(abl).T
abl_table["change in mix"] = abl_table["mix"] - final_score["mix"]
print(abl_table.round(4).to_string())
topic_score = score(under_topic_folds(fit_stack, blocks)["oof"])
print(f"\nthe full stack with topic folds (a whole topic held out at a time): mix {topic_score['mix']:.4f}"
      f" | rmse {topic_score['rmse']:.4f} | pearson {topic_score['pearson']:.4f}")
print("\nstack weights:", {k: round(float(v), 3) for k, v in final["weights"].items()}, "| intercept:", round(final["intercept"], 3))
print(f"{time.time() - t0:.0f} s")

In [ ]:
# 6.3 Plots: predictions against truth, error by score, stack weights, small feature correlations.
p = final["oof"].mean(0)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

ax = axes[0, 0]
ax.scatter(yk + np.random.uniform(-0.08, 0.08, len(yk)), p, s=9, alpha=0.5, c=np.where(short_tr, "tab:orange", "tab:blue"))
ax.plot([1, 5], [1, 5], "k--", lw=1)
ax.set_xlabel("true score (jittered)"); ax.set_ylabel("out of fold prediction")
ax.set_title("Predicted against true (orange = short recordings)")

ax = axes[0, 1]
by = pd.DataFrame({"score": np.round(yk), "error": p - yk}).groupby("score")["error"]
by.mean().plot(kind="bar", yerr=by.std(), ax=ax, color="steelblue", capsize=3)
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("prediction minus truth"); ax.set_title("Error by true score")

ax = axes[1, 0]
pd.Series(final["weights"]).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("weight in the blend"); ax.set_title("How much the stack trusts each block")

ax = axes[1, 1]
corr = F_tr[keep].corrwith(pd.Series(yk, index=F_tr[keep].index)).dropna()
corr = corr.reindex(corr.abs().sort_values().index).tail(12)
corr.plot(kind="barh", ax=ax, color=np.where(corr > 0, "steelblue", "indianred"))
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("correlation with the score"); ax.set_title("Strongest small features")
plt.tight_layout(); plt.show()

In [ ]:
# 6.4 How different is the test set, block by block? A classifier tries to tell training files from test
# files. AUC 0.5 = indistinguishable, 1.0 = perfectly separable. Folds keep each speaker on one side.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

v_all = np.vstack([v_tr, v_te])
_, g_all = connected_components((v_all @ v_all.T) >= 0.80, directed=False)
is_test = np.r_[np.zeros(len(v_tr)), np.ones(len(v_te))]
short_all = np.r_[short_tr, short_te]

def domain_auc(X, mask):
    X, yd, g = X[mask], is_test[mask], g_all[mask]
    splits = list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, yd, g))
    if any(len(np.unique(yd[tr])) < 2 for tr, _ in splits):
        splits = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, yd))
    p = np.zeros(len(yd))
    for tr, va in splits:
        p[va] = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000)).fit(X[tr], yd[tr]).predict_proba(X[va])[:, 1]
    return roc_auc_score(yd, p)

check = {"text bge (before topic neutral)": (T_tr["bge"][keep], T_te["bge"]), "voice fingerprint": (voice_tr[keep], voice_te),
         **{n: (all_train[n][keep], all_test[n]) for n in all_train if n != DEB}}
shift_table = pd.DataFrame({n: {"short recordings": domain_auc(np.vstack([a, b]), short_all),
                                "long recordings": domain_auc(np.vstack([a, b]), ~short_all)}
                            for n, (a, b) in check.items()}).T
print("Training against test, AUC per block")
print(shift_table.round(3).to_string())

## 7. Final model, training RMSE and submission

The block models and the blend are refitted on all training files that were used.

Three error numbers are printed. They answer different questions.

* **Training RMSE (in sample)**: the model predicts the files it was fitted on. This is the number the
  task asks for. It is optimistic.
* For the fine-tuned DeBERTa block no model was trained on all files, so inside this number each
  training file gets DeBERTa's out of fold prediction. That makes it slightly less optimistic.
* **Out of fold, speaker folds**: every file is predicted by models that never heard that speaker.
  This is the honest estimate for new speakers.

In [ ]:
plain = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
fitted_keep = final["predict"]({n: all_train[n][keep] for n in all_train})   # the training files used

print(f"TRAINING RMSE (in sample, {keep.sum()} files used for training): {plain(yk, fitted_keep):.4f}"
      f"   Pearson {pearsonr(yk, fitted_keep)[0]:.4f}")
print(f"Out of fold, speaker folds, unweighted                 : {plain(yk, final['oof'].mean(0)):.4f}"
      f"   Pearson {pearsonr(yk, final['oof'].mean(0))[0]:.4f}")
print(f"Out of fold, speaker folds, weighted to the test mix   : rmse {final_score['rmse']:.4f}"
      f"   Pearson {final_score['pearson']:.4f}   mix {final_score['mix']:.4f}")

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: final["test_pred"]})
assert len(submission) == len(test) and submission[TARGET].notna().all()
submission.to_csv(WORK / "submission.csv", index=False)
print("\nwrote", WORK / "submission.csv", submission.shape)

plt.figure(figsize=(6, 3))
plt.hist(yk, bins=20, alpha=0.5, density=True, label="training labels")
plt.hist(final["test_pred"], bins=20, alpha=0.5, density=True, label="test predictions")
plt.legend(); plt.xlabel("score"); plt.tight_layout(); plt.show()
submission.head()

In [ ]:
# Check: is this the same file that was scored on the leaderboard? (Only runs if that file is attached.)
ref = {1: "submission_c_ft_added.csv", 2: "submission_c3_base_2runs.csv"}[FT_RUNS]
hits = [p for p in [WORK / ref] + [Path(h) for h in glob.glob(f"{INPUT_ROOT}/**/{ref}", recursive=True)] if p.exists()]
if hits:
    old = pd.read_csv(hits[0]).set_index(ID_COL)[TARGET].reindex(submission[ID_COL]).values
    print(f"largest difference from {ref}: {np.abs(old - submission[TARGET].values).max():.6f}")
else:
    print(f"{ref} not attached, nothing to compare")

## 8. Leaderboard history and what did not help

Public leaderboard scores (lower is better; about 130 of the 216 test files).

| Step | Change | Leaderboard |
|---|---|---|
| 1 | Whisper audio embedding, ridge | 0.4362 |
| 2 | Linear recalibration (stacking a single block) | 0.3905 |
| 3 | Stack of audio, text and simple counts | 0.3886 |
| 4 | 34 small features (vocabulary, grammar models, pauses) | 0.3808 |
| 5 | Second text model (bge) | 0.3736 |
| 6 | Qwen2.5 7B judge state | 0.3570 |
| 7 | Text embeddings made topic neutral | 0.3537 |
| 8 | Phi-3.5 mini judge state | 0.3440 |
| 9 | Fine-tuned DeBERTa-v3-base | 0.3358 |

| Idea | Result |
|---|---|
| Keeping the zero-score files in training | Worse on every feature block |
| Weighting training files to the test mix | No net gain: better on short recordings, worse on long |
| SVR (non linear) on the audio embedding | Same in cross validation, worse on the leaderboard (0.4092 against 0.3886) |
| Other Whisper encoder layers | Error falls steadily with depth, the last layers are best |
| WavLM as a second audio block | Better in cross validation, worse on the leaderboard. It leans on recognising voices |
| Audio LLM (Qwen2-Audio) state | Better in cross validation, worse on the leaderboard (0.3652 against 0.3570) |
| The judges' own grades, and their confidence | No weight in the stack. The internal state already holds this |
| Qwen3 8B and Phi-4 mini as extra judges | No change (0.3540, 0.3534 against 0.3537) |
| Asking about errors instead of the rubric | Worse (0.3606) |
| Making Whisper or the judges topic neutral too | Worse (0.3598, 0.3773). Part of what differs between topics is real |
| Stretching or shifting the final predictions | No gain. The blend is already calibrated |
| Verbatim transcripts (Whisper prompted to keep fillers and false starts) | Worse (0.3461 to 0.3571). Fillers did not track the score |
| Judges shown five scored example answers first | No gain (0.3440, 0.3476) |
| DeBERTa-v3-large in place of, or next to, base | Worse (0.3424, 0.3387) |
| A second DeBERTa run averaged in | Better in cross validation, a tie on the leaderboard (0.3362) |
| Qwen2.5 1.5B fine-tuned with LoRA | Weaker than DeBERTa alone, no gain in the stack (0.3378, 0.3479) |

## 9. Limits

* Whisper tidies up speech. It drops fillers and some slips, so part of the grammar signal never
  reaches the transcript.
* Cross validation could not rank the last few choices: the test set answers new questions, which no
  split of the training data fully imitates. Those choices were checked on the public leaderboard,
  which makes its score somewhat optimistic for the hidden 40%.
* The gain from the Phi-3.5 judge is specific to that model. Similar models added nothing, and why it
  helps is not understood.
* One fine-tuning run is noisy: two runs of the same DeBERTa recipe differed by about 0.01 in cross
  validation, as much as some of the differences between ideas.
* Speaker groups and topics are both found with simple thresholds and clustering. Some may be split or
  merged.
* The labels are human ratings in half point steps, so some error cannot be removed.